# Context & Knowledge Patterns

Companion notebook for the [Context & Knowledge lesson](https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/03-context-and-knowledge-patterns).

**The idea in one sentence.** An agent's context window is finite, so it retrieves the
*relevant* knowledge on demand — **RAG**: embed the query and documents, find the
closest chunks by cosine similarity, and stuff only those into the prompt.

The design choices that decide RAG quality:

- **Chunk granularity:** a whole document dilutes the signal; sentence-level chunks let
  the truly relevant passage score much higher.
- **Retrieval scoring:** cosine over embeddings (semantic) vs BM25 (lexical) — the
  exercise builds BM25.

We build a vector store and retriever from scratch, **validate that chunking sharpens
retrieval**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json, re
from collections import Counter
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

BG     = '#0f1117'
CARD   = '#1a1d27'
BRAND  = '#6366f1'
TEAL   = '#2dd4bf'
ROSE   = '#fb7185'
YELLOW = '#fbbf24'
TEXT   = '#e2e8f0'
MUTED  = '#64748b'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'axes.edgecolor': MUTED, 'text.color': TEXT,
    'axes.labelcolor': TEXT, 'xtick.color': TEXT,
    'ytick.color': TEXT, 'grid.color': MUTED, 'grid.alpha': 0.3,
})
np.random.seed(42)
print('Setup complete.')

## 1  Toy knowledge base

We'll use a small collection of documents about AI and distributed systems.

In [ ]:
DOCUMENTS = [
    "Transformer models use self-attention to process sequences in parallel, enabling efficient training at scale.",
    "Retrieval-Augmented Generation (RAG) combines a retriever that finds relevant documents with an LLM that synthesises them.",
    "Kubernetes orchestrates containerised workloads across a cluster, handling scheduling and fault tolerance automatically.",
    "Fine-tuning adapts a pre-trained language model to a specific domain by continuing training on a small labelled dataset.",
    "Vector databases store high-dimensional embeddings and support approximate nearest-neighbour search at low latency.",
    "Prompt engineering is the practice of crafting input text to elicit desired behaviour from a language model.",
    "The attention mechanism computes a weighted sum of value vectors, where weights come from query-key dot products.",
    "Distributed tracing tracks a request as it flows through microservices, capturing latency at each hop.",
    "Low-rank adaptation (LoRA) inserts trainable rank-decomposition matrices into transformer layers for parameter-efficient fine-tuning.",
    "Cosine similarity measures the angle between two vectors: sim(a,b) = dot(a,b) / (norm(a) * norm(b)).",
]

QUERIES = [
    "How does the attention mechanism work?",
    "What is retrieval-augmented generation?",
    "How can I fine-tune a language model efficiently?",
]

print(f'Corpus: {len(DOCUMENTS)} documents')
for i, d in enumerate(DOCUMENTS):
    print(f'  [{i}] {d[:70]}...' if len(d) > 70 else f'  [{i}] {d}')

## 2  Bag-of-words embedding

Real RAG uses dense neural embeddings (e.g. `text-embedding-3-small`).
Here we use a vocabulary-based TF representation so the notebook is
self-contained. The cosine-similarity math is identical.

In [ ]:
STOPWORDS = {'a','an','the','is','are','to','of','in','and','or','at','by',
             'for','with','from','that','this','it','as','on','be','have',
             'has','was','were','do','does','did','not','but','so','its'}

def tokenise(text: str) -> List[str]:
    tokens = re.findall(r'[a-z]+', text.lower())
    return [t for t in tokens if t not in STOPWORDS]

# Build vocabulary from corpus
vocab_counts = Counter()
for doc in DOCUMENTS:
    vocab_counts.update(tokenise(doc))

# Keep the 200 most frequent terms
vocab = sorted([w for w, c in vocab_counts.most_common(200)])
vocab_idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

def embed(text: str) -> np.ndarray:
    """Bag-of-words TF vector (L2-normalised)."""
    vec = np.zeros(V, dtype=float)
    for tok in tokenise(text):
        if tok in vocab_idx:
            vec[vocab_idx[tok]] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

print(f'Vocabulary size: {V}')
sample_emb = embed(DOCUMENTS[0])
print(f'Sample embedding shape: {sample_emb.shape}, nonzero dims: {np.count_nonzero(sample_emb)}')

## 3  Vector store and cosine retrieval

We implement the cosine similarity formula from the lesson:

$$\text{sim}(q, d) = \frac{q \cdot d}{\|q\| \cdot \|d\|}$$

Because our embeddings are already L2-normalised, `sim(q, d) = dot(q, d)`.

In [ ]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

class VectorStore:
    def __init__(self, embed_fn):
        self.embed_fn  = embed_fn
        self.chunks:   List[str]        = []
        self.vectors:  List[np.ndarray] = []

    def add(self, text: str):
        self.chunks.append(text)
        self.vectors.append(self.embed_fn(text))

    def retrieve(self, query: str, k: int = 3) -> List[Tuple[str, float]]:
        q_vec  = self.embed_fn(query)
        scores = [cosine_sim(q_vec, v) for v in self.vectors]
        top_k  = np.argsort(scores)[::-1][:k]
        return [(self.chunks[i], round(scores[i], 4)) for i in top_k]


# Index the corpus
store = VectorStore(embed)
for doc in DOCUMENTS:
    store.add(doc)

print('Retrieval results:')
for query in QUERIES:
    results = store.retrieve(query, k=2)
    print(f'\n  Query: {query!r}')
    for chunk, score in results:
        print(f'    [{score:.4f}] {chunk[:80]}...' if len(chunk) > 80 else f'    [{score:.4f}] {chunk}')

### Validate: the retriever returns the semantically-relevant chunk

The whole point of the vector store is that a query retrieves the *right* document by
cosine similarity, not by keyword luck. We add the knowledge base, query it, and confirm
the top-1 result is the on-topic document — and that cosine of a chunk with itself is 1
(a proper similarity).

In [ ]:
vs_check = VectorStore(embed)
for d in DOCUMENTS:
    vs_check.add(d)
top_chunk, top_score = vs_check.retrieve('How does the attention mechanism work?', k=1)[0]
print(f'top retrieved chunk (score {top_score}): {top_chunk[:70]}...')
assert 'attention' in top_chunk.lower(), 'the retriever should surface the attention document'
# cosine self-similarity is exactly 1
assert abs(cosine_sim(embed(DOCUMENTS[0]), embed(DOCUMENTS[0])) - 1.0) < 1e-6
# retrieval is ranked by descending score
scored = vs_check.retrieve('vector database embeddings', k=len(DOCUMENTS))
assert [s for _, s in scored] == sorted([s for _, s in scored], reverse=True)
print('\n✅ the vector store retrieves the relevant chunk by cosine similarity, ranked by score')

## 4  Effect of chunking granularity

Chunk size affects retrieval precision. Let's compare retrieving at the
sentence level vs. at the document level for a multi-sentence document.

In [ ]:
# A longer document with multiple sentences
long_doc = (
    "The transformer architecture uses multi-head self-attention. "
    "Each attention head independently attends to different positions. "
    "After attention, a feed-forward network processes each position. "
    "Positional encodings are added to provide sequence order information. "
    "The encoder-decoder structure was introduced in the original 2017 paper."
)

# Sentence-level chunks
sentence_chunks = [s.strip() for s in long_doc.split('.') if s.strip()]

store_sentence = VectorStore(embed)
for chunk in sentence_chunks:
    store_sentence.add(chunk)

store_document = VectorStore(embed)
store_document.add(long_doc)

test_query = "What are positional encodings?"
r_sent = store_sentence.retrieve(test_query, k=1)
r_doc  = store_document.retrieve(test_query, k=1)

print(f'Query: {test_query!r}\n')
print(f'Sentence-level top result (score={r_sent[0][1]:.4f}):')
print(f'  "{r_sent[0][0]}"\n')
print(f'Document-level top result (score={r_doc[0][1]:.4f}):')
print(f'  "{r_doc[0][0][:100]}..."')

Sentence-level chunks return the *specific* sentence about positional encodings;
document-level retrieval returns the whole paragraph, which dilutes the
relevance signal. Let's visualise retrieval scores across chunk sizes.

In [ ]:
# Show scores for all sentence chunks vs. the whole doc
scores_sent = [cosine_sim(embed(test_query), embed(c)) for c in sentence_chunks]
score_doc   = cosine_sim(embed(test_query), embed(long_doc))

fig, ax = plt.subplots(figsize=(10, 4))
short_labels = [f'S{i+1}' for i in range(len(sentence_chunks))]
bars = ax.bar(short_labels, scores_sent, color=BRAND, width=0.5, label='sentence chunks')
ax.axhline(score_doc, color=ROSE, ls='--', lw=2, label=f'full document ({score_doc:.4f})')

for bar, sc in zip(bars, scores_sent):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f'{sc:.3f}', ha='center', fontsize=8)

ax.set_xlabel('Chunk'); ax.set_ylabel('Cosine similarity with query')
ax.set_title('Chunk granularity vs. retrieval precision', color=TEXT, pad=8)
ax.legend()

# Annotate the best sentence
best_idx = int(np.argmax(scores_sent))
ax.annotate(f'Best match:\n"{sentence_chunks[best_idx][:40]}..."',
            xy=(best_idx, scores_sent[best_idx]),
            xytext=(best_idx + 0.8, scores_sent[best_idx] + 0.04),
            arrowprops=dict(arrowstyle='->', color=TEAL), color=TEAL, fontsize=8)

plt.tight_layout(); plt.show()

## 5  Prompt/Response Optimiser

Build a RISCO-template prompt builder and a JSON response extractor.

In [ ]:
class PromptOptimiser:
    """Assembles prompts following the RISCO template."""

    def build(
        self,
        role: str,
        instruction: str,
        specification: str,
        context_chunks: List[str],
        output_cue: str = 'JSON:\n',
    ) -> str:
        context_text = '\n\n'.join(
            f'[DOC {i+1}]\n{chunk}' for i, chunk in enumerate(context_chunks)
        )
        return (
            f"ROLE: {role}\n\n"
            f"INSTRUCTION: {instruction}\n\n"
            f"SPECIFICATION: {specification}\n\n"
            f"CONTEXT:\n{context_text}\n\n"
            f"{output_cue}"
        )


class ResponseOptimiser:
    """Extracts structured data from raw FM responses."""

    def extract_json(self, raw: str) -> dict:
        try:
            return json.loads(raw.strip())
        except json.JSONDecodeError:
            pass
        match = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        raise ValueError(f"No valid JSON in: {raw[:200]!r}")

    def validate(self, data: dict, required_keys: List[str]) -> bool:
        return all(k in data for k in required_keys)


# Demo
opt_p = PromptOptimiser()
opt_r = ResponseOptimiser()

query   = QUERIES[0]  # "How does the attention mechanism work?"
context = store.retrieve(query, k=2)

prompt = opt_p.build(
    role='You are a concise ML tutor.',
    instruction=f'Answer the following question based only on the provided context: {query}',
    specification='{"answer": "<1-2 sentence answer>", "source_doc": <doc number>}',
    context_chunks=[chunk for chunk, _ in context],
)

print('=== Optimised Prompt ===')
print(prompt)

# Simulate FM response
mock_response = '{"answer": "The attention mechanism computes a weighted sum of value vectors using query-key dot products as weights.", "source_doc": 2}'
parsed = opt_r.extract_json(mock_response)
print('\n=== Parsed Response ===')
print(parsed)

## 6  End-to-end RAG with the optimiser

In [ ]:
def mock_llm_rag(prompt: str) -> str:
    """Mock LLM: returns a canned answer that mentions retrieval content."""
    if 'retrieval' in prompt.lower() or 'rag' in prompt.lower():
        return json.dumps({'answer': 'RAG combines a retriever with an LLM to ground generation in retrieved documents.', 'source_doc': 1})
    if 'attention' in prompt.lower():
        return json.dumps({'answer': 'Attention computes weighted sums of value vectors using query-key dot products.', 'source_doc': 2})
    return json.dumps({'answer': 'LoRA inserts low-rank matrices to adapt transformer weights with few trainable parameters.', 'source_doc': 3})


def rag_pipeline(query: str, store: VectorStore, k: int = 2) -> dict:
    # Step 1: Retrieve
    results = store.retrieve(query, k=k)
    chunks  = [chunk for chunk, _ in results]
    scores  = [score for _, score in results]

    # Step 2: Build optimised prompt
    prompt = opt_p.build(
        role='You are a concise ML tutor.',
        instruction=f'Answer based only on the context: {query}',
        specification='{"answer": "...", "source_doc": <int>}',
        context_chunks=chunks,
    )

    # Step 3: Call LLM
    raw  = mock_llm_rag(prompt)
    data = opt_r.extract_json(raw)

    return {'query': query, 'answer': data.get('answer'), 'retrieval_scores': scores}


print('End-to-end RAG pipeline:\n')
for q in QUERIES:
    res = rag_pipeline(q, store)
    print(f'Q: {res["query"]}')
    print(f'A: {res["answer"]}')
    print(f'   (top retrieval scores: {res["retrieval_scores"]})')
    print()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **chunks too big** | dilute the relevant signal → lower retrieval scores (verified) |
| **chunks too small** | fragment context; the answer spans multiple chunks |
| **k too large** | floods the prompt with irrelevant context and burns tokens (demo) |
| **embedding mismatch** | query and doc embedded differently → poor recall |
| **stale index** | the knowledge base drifts from the vector store; re-embed on updates |

Demo: retrieval scores drop off fast, so a small k keeps precision high.

In [ ]:
# Retrieval has a precision/recall knob too: k. Too small a k misses the relevant chunk
# (low recall); too large floods the prompt with irrelevant context (low precision, more
# tokens). We show retrieval scores fall off past the top few.
store_demo = VectorStore(embed)
for d in DOCUMENTS:
    store_demo.add(d)
q = 'How do I reset my password?'
results = store_demo.retrieve(q, k=len(DOCUMENTS))
print(f'retrieval scores for "{q}" (ranked):')
for chunk, sc in results:
    print(f'  {sc:.3f}  {chunk[:50]}')
top_scores = [sc for _, sc in results]
assert top_scores == sorted(top_scores, reverse=True), 'retrieval returns chunks in score order'
print('\nScores drop off fast -> a small k keeps precision high; too large a k adds noise + tokens.')

## ✏️ Your turn

### Exercise A — Implement `top_k_retrieval` with a score threshold

Extend the `VectorStore.retrieve()` method by adding a `min_score` parameter
that filters out any retrieved chunks whose cosine similarity is below the
threshold, even if they are in the top-k.

In [ ]:
class FilteredVectorStore(VectorStore):
    def retrieve(self, query: str, k: int = 3,
                 min_score: float = 0.0) -> List[Tuple[str, float]]:
        # TODO(you): call super().retrieve(query, k), then filter by min_score
        pass


fstore = FilteredVectorStore(embed)
for doc in DOCUMENTS:
    fstore.add(doc)

results_high = fstore.retrieve(QUERIES[0], k=5, min_score=0.5)
results_low  = fstore.retrieve(QUERIES[0], k=5, min_score=0.0)

assert len(results_high) <= len(results_low), 'high threshold should return ≤ results'
assert all(s >= 0.5 for _, s in results_high), 'all returned scores should be >= min_score'
print(f'k=5, min_score=0.5 → {len(results_high)} results')
print(f'k=5, min_score=0.0 → {len(results_low)} results')
print('passed ✓')

### Exercise B — Implement BM25 scoring

Implement the BM25 score for a single (query, document) pair.
Use $k_1 = 1.5$ and $b = 0.75$, and compute IDF as
$\text{IDF}(t) = \ln\left(\frac{N - n_t + 0.5}{n_t + 0.5} + 1\right)$
where $N$ is the number of documents and $n_t$ is the document frequency of term $t$.

In [ ]:
def bm25_score(query: str, document: str, corpus: List[str],
               k1: float = 1.5, b: float = 0.75) -> float:
    # TODO(you): implement BM25
    # Hints:
    #   - tokenise query and document
    #   - compute average document length from corpus
    #   - for each query term, compute IDF and term-frequency component
    #   - sum up the per-term scores
    pass


score = bm25_score('attention mechanism', DOCUMENTS[6], DOCUMENTS)
assert score > 0, 'BM25 score should be positive for relevant document'

irrelevant_score = bm25_score('attention mechanism', DOCUMENTS[2], DOCUMENTS)
assert score > irrelevant_score, 'relevant doc should score higher than irrelevant'
print(f'BM25 (attention doc):    {score:.4f}')
print(f'BM25 (kubernetes doc):   {irrelevant_score:.4f}')
print('passed ✓')

<details><summary>Solution — Exercise A</summary>

```python
class FilteredVectorStore(VectorStore):
    def retrieve(self, query, k=3, min_score=0.0):
        raw = super().retrieve(query, k)
        return [(chunk, score) for chunk, score in raw if score >= min_score]
```

</details>

<details><summary>Solution — Exercise B</summary>

```python
def bm25_score(query, document, corpus, k1=1.5, b=0.75):
    import math
    N       = len(corpus)
    avgdl   = np.mean([len(tokenise(d)) for d in corpus])
    doc_tok = tokenise(document)
    q_tok   = tokenise(query)
    dl      = len(doc_tok)
    tf_map  = Counter(doc_tok)

    score = 0.0
    for term in set(q_tok):
        nt  = sum(1 for d in corpus if term in tokenise(d))
        idf = math.log((N - nt + 0.5) / (nt + 0.5) + 1)
        tf  = tf_map.get(term, 0)
        tf_norm = tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl / avgdl))
        score += idf * tf_norm
    return score
```

</details>

## Key takeaways

- **RAG retrieves relevant knowledge on demand** instead of stuffing everything into a
  finite context window.
- **Chunk granularity matters:** sentence chunks let the relevant passage outscore the
  diluted whole-document average (verified).
- **k trades precision for recall:** too small misses the answer, too large floods the
  prompt with noise and tokens (demo).
- **Lexical (BM25) and semantic (embeddings) retrieval are complementary** — the
  exercise builds BM25; hybrid retrieval combines both.